# Roomscape: Colab Reconstruction

3D Gaussian Splatting digital twin pipeline on Google Colab (extract frames -> COLMAP -> Gaussian splat -> collision -> export).
Use a GPU runtime (Runtime > Change runtime type > T4/L4/A100).

In [ ]:
!nvidia-smi
!apt-get -qq install -y colmap ffmpeg  # apt COLMAP is CPU-only

In [ ]:
!pip -q install nerfstudio pyyaml matplotlib pytest

In [ ]:
# Clone or update the Roomscape repository
REPO_URL = "https://github.com/s2kumar2007/roomscape"
from pathlib import Path

if not Path("/content/roomscape").exists():
    !git clone {REPO_URL} /content/roomscape

%cd /content/roomscape
!git pull
!pytest -q

In [ ]:
# Recommended: load input video from Google Drive
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")

DRIVE_VIDEO = "/content/drive/MyDrive/roomscape/room.mp4"
input_dir = Path("./input")
input_dir.mkdir(parents=True, exist_ok=True)

src = Path(DRIVE_VIDEO)
if src.exists():
    shutil.copy(src, input_dir / src.name)
    print(f"Copied {src} to {input_dir / src.name}")
else:
    print(f"Video not found at {src}. Check the path or use the upload cell below.")

In [ ]:
# Optional: upload video from local browser
from google.colab import files
import shutil
from pathlib import Path

input_dir = Path("./input")
input_dir.mkdir(parents=True, exist_ok=True)

up = files.upload()
for filename in up.keys():
    shutil.move(filename, input_dir / filename)
    print(f"Saved {filename} to {input_dir / filename}")

In [ ]:
# Configure offscreen platform and switch COLMAP to CPU
import os
import re
from pathlib import Path

os.environ["QT_QPA_PLATFORM"] = "offscreen"

cfg_file = Path("pipeline/config.yaml")
content = cfg_file.read_text(encoding="utf-8")
new_content, count = re.subn(r"use_gpu:\s*true", "use_gpu: false", content)
assert count > 0, "use_gpu: true not found in pipeline/config.yaml"
assert "use_gpu: false" in new_content, "Replacement failed"
cfg_file.write_text(new_content, encoding="utf-8")

print("Set QT_QPA_PLATFORM=offscreen and switched COLMAP to CPU in pipeline/config.yaml")

In [ ]:
# Settings
FRAMES = 300
TRIM_START = 0
TRIM_END = 0  # for a long video try 100
MODE = "fast"

FLAGS = f"--mode {MODE} --frames {FRAMES} --trim-start {TRIM_START} --trim-end {TRIM_END}"

!python pipeline/run_pipeline.py --video input --out out {FLAGS}

In [ ]:
# Resume execution if the pipeline was interrupted
!python pipeline/run_pipeline.py --video input --out out {FLAGS} --resume

In [ ]:
# Display results report and visualizations
from IPython.display import display, Markdown, Image
from pathlib import Path

report_md = Path("out/report.md")
if report_md.exists():
    display(Markdown(report_md.read_text(encoding="utf-8")))

for img_path in [Path("out/debug/floor_alignment.png"), Path("out/occupancy.png")]:
    if img_path.exists():
        print(f"\n{img_path.name}:")
        display(Image(str(img_path)))

In [ ]:
# Package and download lightweight results bundle
import shutil
from pathlib import Path
from google.colab import files

result_dir = Path("./result")
if result_dir.exists():
    shutil.rmtree(result_dir)
result_dir.mkdir(parents=True, exist_ok=True)

out_dir = Path("out")
files_to_copy = [
    "report.json",
    "report.md",
    "timings.json",
    "frames_report.json",
    "reconstruct_report.json",
    "transform.json",
    "collision.json",
    "occupancy.png",
    "export_report.json",
    "cameras_aligned.json",
]

for fname in files_to_copy:
    src_file = out_dir / fname
    if src_file.exists():
        shutil.copy(src_file, result_dir / fname)

for folder_name in ["scene", "debug"]:
    src_folder = out_dir / folder_name
    if src_folder.exists():
        shutil.copytree(src_folder, result_dir / folder_name, dirs_exist_ok=True)

shutil.make_archive("roomscape_result", "zip", "./result")
files.download("roomscape_result.zip")